In [1]:
import os

In [2]:
%pwd

'd:\\Python AI ML DL\\ML Flow e-t-e\\E-to-E-ML-Project-using-ML-Flow\\research'

In [2]:
os.chdir("d:\\Python AI ML DL\\ML Flow e-t-e\\E-to-E-ML-Project-using-ML-Flow")

In [4]:
%pwd

'D:\\Python AI ML DL\\ML Flow e-t-e\\E-to-E-ML-Project-using-ML-Flow\\research'

In [3]:
from dataclasses import dataclass
from mlProject.config.configuration import ConfigurationManager 
from pathlib import Path
from mlProject import logger    

@dataclass(frozen=True)
class DataTransformationConfig:
    train_data_path: Path
    test_data_path: Path
    root_dir: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [4]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories
from mlProject.entity.config_entity import ModelTrainerConfig

class ConfigurationManager:
    def __init__(self, 
                 config_file_path = CONFIG_FILE_PATH, 
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.model_trainer
        
        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column="quality",
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path
        )
        
        return model_trainer_config

In [5]:
import pandas as pd 
import os 
from mlProject import logger
from sklearn.linear_model import ElasticNet
import joblib

In [6]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop(columns=[self.config.target_column])
        test_x = test_data.drop(columns=[self.config.target_column])
        train_y = train_data[self.config.target_column]
        test_y = test_data[self.config.target_column]

        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)

        score = lr.score(test_x, test_y)
        logger.info(f"Model training completed. Test R² score: {score}")

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

In [7]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    logger.exception(e)

[2026-04-20 18:22:07,070: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-20 18:22:07,077: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-20 18:22:07,082: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-20 18:22:07,085: INFO: common: created directory at: artifacts]
[2026-04-20 18:22:07,088: INFO: common: created directory at: artifacts/model_training]
[2026-04-20 18:22:07,184: INFO: 3118823702: Model training completed. Test R² score: 0.08785680512866556]


In [9]:
# Test config loading
test_config = ConfigurationManager()
test_model_config = test_config.get_model_trainer_config()
print(f"Root dir: {test_model_config.root_dir}")
print(f"Model name: {test_model_config.model_name}")
print(f"Train data path: {test_model_config.train_data_path}")
print(f"Test data path: {test_model_config.test_data_path}")
print(f"Target column: {test_model_config.target_column}")
print(f"Alpha: {test_model_config.alpha}")
print(f"L1 ratio: {test_model_config.l1_ratio}")

[2026-04-20 18:20:26,030: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-20 18:20:26,040: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-20 18:20:26,047: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-20 18:20:26,052: INFO: common: created directory at: artifacts]
[2026-04-20 18:20:26,056: INFO: common: created directory at: artifacts/model_training]
Root dir: artifacts/model_training
Model name: model.joblib
Train data path: artifacts/data_transformation/train.csv
Test data path: artifacts/data_transformation/test.csv
Target column: quality
Alpha: 0.5
L1 ratio: 0.7
